In [ ]:
!pip install python-dotenv

In [ ]:
import os
import json
import time
from datetime import datetime
from pathlib import Path
from openai import OpenAI
import pandas as pd

from dotenv import load_dotenv

In [ ]:
# Configuration
MODEL = "gpt-5.4-mini"  # OpenAI model
TWEETS_PER_CELL = 100   # Start small; change to 1200 for full run
OUTPUT_DIR = Path("synthetic_data")
OUTPUT_DIR.mkdir(exist_ok=True)

# Targets and stances (P-Stance schema)
TARGETS = ["Donald Trump", "Joe Biden", "Bernie Sanders"]
STANCES = ["FAVOR", "AGAINST"]

# Initialize OpenAI client
load_dotenv(Path(__file__).resolve().parent / ".env")
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print(f"Configured for {MODEL}")
print(f"Will generate {TWEETS_PER_CELL} tweets per cell")
print(f"Total cells: {len(TARGETS) * len(STANCES)} = {len(TARGETS

Prompt Template

In [ ]:
def build_prompt(target: str, stance: str) -> str:
    """
    Build a neutral, label-only prompt for synthetic tweet generation.
    
    Deliberately minimal to simulate naive practitioner usage.
    Any bias in output reflects the model's internalized priors,
    not researcher prompt engineering.
    """
    stance_phrase = "in favor of" if stance == "FAVOR" else "against"
    
    prompt = (
        f"Write a single tweet that expresses a stance {stance_phrase} {target}. "
        f"The tweet should sound like a real Twitter user. "
        f"Keep it under 280 characters. "
        f"Return only the tweet text, no quotation marks, no explanations, no hashtags-only posts."
    )
    return prompt


# Quick test
print(build_prompt("Donald Trump", "FAVOR"))
print("\n")
print(build_prompt("Joe Biden", "AGAINST"))

Tweet Generation Function

In [ ]:
def generate_tweet(target: str, stance: str, max_retries: int = 3) -> dict:
    """
    Generate a single synthetic tweet for a given target and stance.
    Returns a dict with metadata for logging.
    """
    prompt = build_prompt(target, stance)
    
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.9,           # High temperature for diversity
                max_completion_tokens=100, # Tweets are short
            )
            
            tweet_text = response.choices[0].message.content.strip()
            
            # Detect refusal (model declined to generate)
            refusal_keywords = ["i can't", "i cannot", "i'm not able", "i won't", 
                                "as an ai", "i'm sorry", "i apologize"]
            refused = any(kw in tweet_text.lower()[:80] for kw in refusal_keywords)
            
            return {
                "target": target,
                "stance": stance,
                "tweet": tweet_text,
                "refused": refused,
                "model": MODEL,
                "prompt": prompt,
                "timestamp": datetime.now().isoformat(),
                "attempt": attempt + 1,
            }
            
        except Exception as e:
            print(f"  Error on attempt {attempt + 1}: {e}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)  # Exponential backoff
            else:
                return {
                    "target": target,
                    "stance": stance,
                    "tweet": None,
                    "refused": False,
                    "error": str(e),
                    "model": MODEL,
                    "prompt": prompt,
                    "timestamp": datetime.now().isoformat(),
                    "attempt": attempt + 1,
                }


# Quick sanity check — generate ONE tweet to verify everything works
test_result = generate_tweet("Donald Trump", "FAVOR")
print(json.dumps(test_result, indent=2))

Batch Generation

In [ ]:
from tqdm.notebook import tqdm

def generate_batch(target: str, stance: str, n: int) -> list:
    """Generate n tweets for a (target, stance) cell with progress bar."""
    results = []
    for i in tqdm(range(n), desc=f"{target} / {stance}", leave=False):
        result = generate_tweet(target, stance)
        results.append(result)
    return results


# Generate all cells
all_results = []
total_cells = len(TARGETS) * len(STANCES)

for i, target in enumerate(TARGETS):
    for j, stance in enumerate(STANCES):
        cell_num = i * len(STANCES) + j + 1
        print(f"\n[{cell_num}/{total_cells}] Generating {TWEETS_PER_CELL} tweets: {target} / {stance}")
        
        batch = generate_batch(target, stance, TWEETS_PER_CELL)
        all_results.extend(batch)
        
        # Save incrementally after each cell (in case of crash)
        df = pd.DataFrame(all_results)
        df.to_csv(OUTPUT_DIR / f"gpt54mini_pilot_progress.csv", index=False)
        print(f"  Saved {len(all_results)} tweets so far")

print(f"\n✅ Done. Generated {len(all_results)} total tweets.")

Save Output

In [ ]:
# Save final clean version with timestamp
timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = OUTPUT_DIR / f"gpt54mini_synthetic_{TWEETS_PER_CELL}per_cell_{timestamp_str}.csv"

df = pd.DataFrame(all_results)
df.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")
print(f"\nDataset shape: {df.shape}")
print(f"\nClass balance:")
print(df.groupby(['target', 'stance']).size())

print(f"\nRefusal rate: {df['refused'].sum()} / {len(df)} ({100*df['refused'].mean():.1f}%)")

# Show 2 sample tweets per cell
print("\n\nSample tweets per cell:")
for target in TARGETS:
    for stance in STANCES:
        samples = df[(df['target']==target) & (df['stance']==stance) & (~df['refused'])]['tweet'].head(2).tolist()
        print(f"\n{target} / {stance}:")
        for s in samples:
            print(f"  • {s[:150]}")

Quality Check

In [ ]:
# Quality diagnostics — what does the data look like?

print("=" * 60)
print("QUALITY DIAGNOSTICS")
print("=" * 60)

# 1. Length distribution
df['tweet_len'] = df['tweet'].fillna('').str.len()
print("\n1. Tweet length stats:")
print(df.groupby(['target', 'stance'])['tweet_len'].describe()[['mean', 'min', 'max']])

# 2. Refusal rates per cell (asymmetric refusals = bias signal!)
print("\n2. Refusal rate per cell:")
refusal_table = df.groupby(['target', 'stance'])['refused'].agg(['sum', 'mean'])
print(refusal_table)

# 3. Duplicate detection
print("\n3. Exact duplicate tweets:")
duplicates = df[df.duplicated(subset='tweet', keep=False)]
print(f"  {len(duplicates)} duplicate tweets")

# 4. Quick visual inspection
print("\n4. Random sample of 10 tweets:")
for _, row in df[~df['refused']].sample(min(10, len(df))).iterrows():
    print(f"  [{row['target']} / {row['stance']}] {row['tweet'][:120]}")